# 05 Feature Extraction

This notebook implements a **session-level behavioural feature extraction pipeline** for BotPrint.

**Objective:** transform cleaned interaction events into meaningful, interpretable session-level behavioural features.

**Input:** `dataset/processed/cleaned_events.csv` (original raw dataset is not modified).

**Output:** `dataset/processed/session_features.csv` and `reports/analysis/feature_dictionary.csv`.

**Key requirements followed:**
- Events grouped by `SessionID`; one output row per session.
- Events sorted chronologically before sequential features.
- Valid observations only; insufficient observations handled explicitly.
- Unavailable features are **not** replaced with zero unless zero is the actual measured value (e.g., 0 navigation events observed).
- Availability indicators (`*_has_data`, `*_event_count`) included where useful.
- `UserType` and `BotType` preserved as separate labels; identifiers and target labels excluded from behavioural features.
- No machine learning performed.

In [1]:
# 1. Setup
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import warnings

warnings.filterwarnings('ignore', category=FutureWarning)

# Resolve project root
_here = Path.cwd()
_root = _here
for _ in range(5):
    if (_root / 'README.md').exists() and (_root / 'dataset').exists():
        break
    _root = _root.parent
PROJECT_ROOT = _root
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from feature_extraction import (
    extract_session_features,
    validate_session_features,
    build_feature_dictionary,
    get_feature_columns,
    ID_COL,
    LABEL_COLS,
)

DATA_PATH = PROJECT_ROOT / 'dataset' / 'processed' / 'cleaned_events.csv'
OUT_PATH = PROJECT_ROOT / 'dataset' / 'processed' / 'session_features.csv'
DICT_PATH = PROJECT_ROOT / 'reports' / 'analysis' / 'feature_dictionary.csv'
ANALYSIS_DIR = PROJECT_ROOT / 'reports' / 'analysis'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Input exists:', DATA_PATH.exists())


Project root: C:\My KS\Research\BotPrint-Research
Input exists: True


In [2]:
# 2. Load cleaned event dataset
df = pd.read_csv(DATA_PATH)
df['Timestamp_dt'] = pd.to_datetime(df.get('Timestamp_dt'), errors='coerce')
print('Event rows:', len(df))
print('Unique sessions:', df['SessionID'].nunique())
print('Columns:', len(df.columns))
print('\nEvent types:')
print(df['EventType'].value_counts().to_string())


Event rows: 1196
Unique sessions: 24
Columns: 36

Event types:
EventType
keypress             460
mousemove            393
click                109
field_interaction    104
scroll                82
session_start         26
form_submit           14
session_end            8


## 3. Feature categories and definitions

Features are computed per session from appropriate event types. Sequential features (e.g., movement distance, direction changes) require events sorted by `Timestamp` within each session.

| Category | Features | Source event types |
|---|---|---|
| Mouse behaviour | mean/std speed, mean/std acceleration, total movement distance, movement frequency, direction changes | `mousemove` |
| Keyboard behaviour | mean/std inter-key interval, typing frequency, typing count | `keypress`/`keydown`/`keyup` |
| Click behaviour | mean/std click interval, click frequency, click count | `click` |
| Scroll behaviour | mean/std scroll speed, mean scroll delta, scroll count | `scroll`/`wheel` |
| Session behaviour | session duration, mean idle time, total event count, event frequency, navigation count | all session events (+ navigation) |

**Missing-value policy:** if a session has insufficient valid observations for a feature, the value is `NaN`. It is not set to zero unless zero is the actual measured value (e.g., navigation count is 0 when no navigation events were observed).

In [3]:
# 4. Extract session-level features
features = extract_session_features(df)
print('Feature table shape:', features.shape)
print('Sessions (rows):', features[ID_COL].nunique())
print('Columns:', list(features.columns))
features.head()


Feature table shape: (24, 38)
Sessions (rows): 24
Columns: ['SessionID', 'UserType', 'BotType', 'CoordinationGroupID', 'SessionSource', 'BotID', 'session_duration_s', 'session_total_events', 'session_event_frequency', 'session_mean_idle_ms', 'session_navigation_count', 'mouse_event_count', 'mouse_has_data', 'mouse_mean_speed', 'mouse_std_speed', 'mouse_mean_acceleration', 'mouse_std_acceleration', 'mouse_total_distance', 'mouse_movement_frequency', 'mouse_direction_changes', 'keyboard_event_count', 'keyboard_has_data', 'keyboard_mean_interkey_ms', 'keyboard_std_interkey_ms', 'keyboard_typing_frequency', 'keyboard_typing_count', 'click_event_count', 'click_has_data', 'click_mean_interval_ms', 'click_std_interval_ms', 'click_frequency', 'click_count', 'scroll_event_count', 'scroll_has_data', 'scroll_mean_speed', 'scroll_std_speed', 'scroll_mean_delta', 'scroll_count']


,SessionID,UserType,BotType,CoordinationGroupID,SessionSource,BotID,session_duration_s,session_total_events,session_event_frequency,session_mean_idle_ms,...,click_mean_interval_ms,click_std_interval_ms,click_frequency,click_count,scroll_event_count,scroll_has_data,scroll_mean_speed,scroll_std_speed,scroll_mean_delta,scroll_count
0,S-0108f7d0-0f85-4c15-a643-b0d53c0da2d8,Human,NaN,NaN,Manual,NaN,38.104,118,3.096788,309.779661,...,4473.571429,5074.442001,0.209952,8.0,1,1,66.399994,0.000000,66.399994,1.0
1,S-1710524c-4560-4abd-906c-6f336edff5de,Bot,CoordinatedBot,coord-f8d0256724,Selenium,bot-02-f02697,31.078,56,1.801918,510.696429,...,2320.166667,2957.473206,0.321771,10.0,2,1,49.002278,69.293242,119.000000,2.0
2,S-1f44b887-5413-4283-91dc-5ac39b63feff,Bot,RandomBot,coord-33f0cbe678,Selenium,bot-02-3d3eda,18.660,12,0.643087,168.333333,...,NaN,NaN,0.053591,1.0,1,1,0.104732,0.000000,66.400024,1.0
3,S-2510ce7a-5d0f-479d-89e2-a251a2e9a162,Bot,RandomBot,coord-33f0cbe678,Selenium,bot-03-142623,20.055,22,1.096983,215.318182,...,1109.500000,263.750829,0.199452,4.0,0,0,NaN,NaN,NaN,0.0
4,S-28ae68f8-c6e5-44f9-8120-1385f632233c,Bot,RandomBot,coord-33f0cbe678,Selenium,bot-04-8cc701,18.770,18,0.958977,214.777778,...,NaN,NaN,0.053277,1.0,1,1,0.143723,0.000000,66.400024,1.0


## 5. Validation checks

Each session must produce exactly one output row, and all source sessions must be represented.

In [4]:
# 5. Validation
try:
    validate_session_features(features, df)
    print('Validation passed: each session has exactly one output row.')
except AssertionError as e:
    print('Validation FAILED:', e)
    raise

# Additional checks
assert features[ID_COL].isna().sum() == 0, 'Null SessionID in output'
print('No null SessionIDs:', True)
print('Source sessions:', df[ID_COL].nunique())
print('Output sessions:', features[ID_COL].nunique())
print('Output rows:', len(features))
print('One row per session:', len(features) == features[ID_COL].nunique() == df[ID_COL].nunique())


Validation passed: each session has exactly one output row.
No null SessionIDs: True
Source sessions: 24
Output sessions: 24
Output rows: 24
One row per session: True


## 6. Labels and feature separation

`UserType` and `BotType` are preserved as labels for later supervised evaluation only. They are **not** behavioural features and must not be used as model inputs. Identifiers and coordination labels are also excluded from the behavioural feature set.

In [5]:
# 6. Separate labels from behavioural features
label_present = [c for c in LABEL_COLS + ['CoordinationGroupID','SessionSource','BotID'] if c in features.columns]
behaviour_cols = get_feature_columns(features)
print('Preserved label/metadata columns:', label_present)
print('Behavioural feature columns (count):', len(behaviour_cols))
print('Behavioural feature columns:', behaviour_cols)

# Sanity: no label leakage into behavioural feature list
leak = [c for c in behaviour_cols if c.lower() in ('usertype','bottype','sessionid','coordinationgroupid','botid')]
print('Label/ID leakage in behavioural features:', leak)
assert not leak, f'Leakage detected: {leak}'


Preserved label/metadata columns: ['UserType', 'BotType', 'CoordinationGroupID', 'SessionSource', 'BotID']
Behavioural feature columns (count): 24
Behavioural feature columns: ['session_duration_s', 'session_total_events', 'session_event_frequency', 'session_mean_idle_ms', 'session_navigation_count', 'mouse_mean_speed', 'mouse_std_speed', 'mouse_mean_acceleration', 'mouse_std_acceleration', 'mouse_total_distance', 'mouse_movement_frequency', 'mouse_direction_changes', 'keyboard_mean_interkey_ms', 'keyboard_std_interkey_ms', 'keyboard_typing_frequency', 'keyboard_typing_count', 'click_mean_interval_ms', 'click_std_interval_ms', 'click_frequency', 'click_count', 'scroll_mean_speed', 'scroll_std_speed', 'scroll_mean_delta', 'scroll_count']
Label/ID leakage in behavioural features: []


## 7. Missing values and availability

Features may be `NaN` when a session lacks the relevant event type or valid observations. Availability indicators show where data exists.

In [6]:
# 7. Missingness / availability summary
avail_cols = [c for c in features.columns if c.endswith('_has_data') or c.endswith('_event_count')]
summary = []
for c in behaviour_cols:
    s = features[c]
    summary.append({
        'feature': c,
        'n_valid': int(s.notna().sum()),
        'n_missing': int(s.isna().sum()),
        'pct_missing': round(float(s.isna().mean() * 100), 2),
        'dtype': str(s.dtype),
    })
miss = pd.DataFrame(summary).sort_values(['pct_missing','feature'], ascending=[False, True])
miss.to_csv(ANALYSIS_DIR / 'session_features_missingness.csv', index=False)
print(miss.to_string(index=False))

if avail_cols:
    print('\nAvailability indicators (sum = sessions with data):')
    print(features[avail_cols].sum().to_string())


                  feature  n_valid  n_missing  pct_missing   dtype
   click_mean_interval_ms       14         10        41.67 float64
    click_std_interval_ms       14         10        41.67 float64
        scroll_mean_delta       14         10        41.67 float64
        scroll_mean_speed       14         10        41.67 float64
         scroll_std_speed       14         10        41.67 float64
  mouse_direction_changes       15          9        37.50 float64
          click_frequency       18          6        25.00 float64
  mouse_mean_acceleration       18          6        25.00 float64
         mouse_mean_speed       18          6        25.00 float64
 mouse_movement_frequency       18          6        25.00 float64
   mouse_std_acceleration       18          6        25.00 float64
          mouse_std_speed       18          6        25.00 float64
     mouse_total_distance       18          6        25.00 float64
keyboard_mean_interkey_ms       19          5        20.83 flo

## 8. Feature dictionary

Every extracted feature is documented with its name, description, formula, source columns, applicable event types, and missing-value handling.

In [7]:
# 8. Feature dictionary
fdict = build_feature_dictionary()
fdict.to_csv(DICT_PATH, index=False)
print('Saved feature dictionary:', DICT_PATH)
print('Documented features:', len(fdict))
fdict


Saved feature dictionary: C:\My KS\Research\BotPrint-Research\reports\analysis\feature_dictionary.csv
Documented features: 32


,feature_name,description,formula,source_columns,applicable_event_types,missing_value_handling
0,mouse_mean_speed,Mean mouse movement speed,mean(Speed) over mousemove events with valid S...,"Speed, EventType, SessionID, Timestamp",mousemove,NaN if no mousemove events with valid Speed
1,mouse_std_speed,Standard deviation of mouse movement speed,"std(Speed, ddof=1) over mousemove events with ...","Speed, EventType, SessionID, Timestamp",mousemove,NaN if no valid Speed observations
2,mouse_mean_acceleration,Mean mouse acceleration,mean(Acceleration) over mousemove events with ...,"Acceleration, EventType, SessionID, Timestamp",mousemove,NaN if no valid observations
3,mouse_std_acceleration,Standard deviation of mouse acceleration,"std(Acceleration, ddof=1); 0 if exactly 1 obse...","Acceleration, EventType, SessionID, Timestamp",mousemove,NaN if no valid observations
4,mouse_total_distance,Total mouse movement distance (Euclidean path ...,sum(sqrt((X_t-X_{t-1})^2 + (Y_t-Y_{t-1})^2)) o...,"X, Y, EventType, SessionID, Timestamp",mousemove,NaN if <2 valid positions and <1 position; 0.0...
5,mouse_movement_frequency,Mouse movement events per second of session,count(mousemove events) / session_duration_s,"EventType, SessionDurationMs/ElapsedMs, Timest...",mousemove + session duration,NaN if no mousemove events or duration unavail...
6,mouse_direction_changes,Number of significant direction changes in mou...,count of consecutive angle differences > 30 de...,"DX, DY, EventType, SessionID, Timestamp",mousemove,NaN if <2 valid movement vectors
7,mouse_event_count,Count of mousemove events in session (availabi...,count(EventType in mouse types),"EventType, SessionID",mousemove,Always 0 or positive integer (measured count)
8,mouse_has_data,Indicator: session has mouse data,1 if mouse_event_count>0 else 0,"EventType, SessionID",mousemove,Always 0 or 1
9,keyboard_mean_interkey_ms,Mean inter-key interval,mean(InterKeyIntervalMs) over keyboard events ...,"InterKeyIntervalMs, EventType, SessionID, Time...",keypress/keydown/keyup,NaN if no valid observations


## 9. Save session-level feature dataset

The final table contains one row per session, with preserved labels and behavioural features. It is saved to `dataset/processed/session_features.csv`.

In [8]:
# 9. Save outputs
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
features.to_csv(OUT_PATH, index=False)
print('Saved:', OUT_PATH)
print('Shape:', features.shape)

# Quick descriptive check (no modelling)
desc = features[behaviour_cols].describe().T
desc.to_csv(ANALYSIS_DIR / 'session_features_descriptive_stats.csv')
print('\nDescriptive stats saved to reports/analysis/session_features_descriptive_stats.csv')
desc.head(15)


Saved: C:\My KS\Research\BotPrint-Research\dataset\processed\session_features.csv
Shape: (24, 38)

Descriptive stats saved to reports/analysis/session_features_descriptive_stats.csv


,count,mean,std,min,25%,50%,75%,max
session_duration_s,24.0,19.576875,15.414623,0.000000,9.027750e+00,18.632500,30.496000,67.328000
session_total_events,24.0,49.833333,62.902731,1.000000,1.175000e+01,18.500000,70.000000,281.000000
session_event_frequency,23.0,2.673305,2.453040,0.515796,1.014184e+00,1.801918,3.611485,9.090909
session_mean_idle_ms,24.0,236.285330,196.221476,0.000000,1.128948e+02,173.910256,277.132415,918.152778
session_navigation_count,24.0,0.583333,1.471960,0.000000,0.000000e+00,0.000000,1.000000,7.000000
mouse_mean_speed,18.0,0.204672,0.261486,0.000000,2.315330e-02,0.072531,0.316884,0.847101
mouse_std_speed,18.0,0.306391,0.437099,0.000000,7.222122e-03,0.053337,0.546034,1.323753
mouse_mean_acceleration,18.0,0.000327,0.000965,-0.001463,-6.014278e-07,0.000008,0.000331,0.002745
mouse_std_acceleration,18.0,0.002523,0.004435,0.000000,5.248422e-07,0.000035,0.004428,0.015918
mouse_total_distance,18.0,1464.695805,2356.426385,0.000000,2.268746e+02,592.320441,1289.286632,7976.609989


## 10. Summary

The pipeline converts cleaned event-level records into one session-level behavioural feature row per `SessionID`. It covers mouse, keyboard, click, scroll, and session behaviours; preserves labels separately; excludes identifiers/target labels from features; documents every feature; and validates that output cardinality matches the number of source sessions. No machine learning is performed in this notebook.